In [12]:
import pandas as pd
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor,ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tabulate import tabulate
import joblib
import time

In [3]:
df=pd.read_csv("walmart.csv")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [5]:
df.drop(columns=["Date"], inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Weekly_Sales  6435 non-null   float64
 2   Holiday_Flag  6435 non-null   int64  
 3   Temperature   6435 non-null   float64
 4   Fuel_Price    6435 non-null   float64
 5   CPI           6435 non-null   float64
 6   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2)
memory usage: 352.0 KB


# Splitting the dataset into features and target variable


In [7]:
X = df.drop('Weekly_Sales', axis=1)
y = df['Weekly_Sales']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# VOTING REGRESSOR
voting_reg = VotingRegressor(
    estimators=[
        ('lr', LinearRegression()),
        ('ridge', Ridge()),
        ('dt', DecisionTreeRegressor(random_state=42))
    ]
)

# STACKING REGRESSOR
base_learners = [
    ('lr', LinearRegression()),
    ('dt', DecisionTreeRegressor(max_depth=5, random_state=42)),
    ('svr', SVR())
]

stacking_reg = StackingRegressor(
    estimators=base_learners,
    final_estimator=LinearRegression()
)

# BAGGING REGRESSOR (Extra Trees)
bagging_reg = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

print("All three ensemble methods initialized")

All three ensemble methods initialized


# Train All Models and Measure Training Time (Computational Complexity)

In [13]:
# Dictionary to store results
results = {}

# VOTING - Training
start_time = time.time()
voting_reg.fit(x_train, y_train)
voting_train_time = time.time() - start_time

y_pred_voting = voting_reg.predict(x_test)
voting_train_mse = mean_squared_error(y_train, voting_reg.predict(x_train))
voting_test_mse = mean_squared_error(y_test, y_pred_voting)
voting_test_r2 = r2_score(y_test, y_pred_voting)
voting_test_mae = mean_absolute_error(y_test, y_pred_voting)

results['Voting'] = {
    'train_time': voting_train_time,
    'train_mse': voting_train_mse,
    'test_mse': voting_test_mse,
    'test_r2': voting_test_r2,
    'test_mae': voting_test_mae
}

# STACKING - Training
start_time = time.time()
stacking_reg.fit(x_train, y_train)
stacking_train_time = time.time() - start_time

y_pred_stacking = stacking_reg.predict(x_test)
stacking_train_mse = mean_squared_error(y_train, stacking_reg.predict(x_train))
stacking_test_mse = mean_squared_error(y_test, y_pred_stacking)
stacking_test_r2 = r2_score(y_test, y_pred_stacking)
stacking_test_mae = mean_absolute_error(y_test, y_pred_stacking)

results['Stacking'] = {
    'train_time': stacking_train_time,
    'train_mse': stacking_train_mse,
    'test_mse': stacking_test_mse,
    'test_r2': stacking_test_r2,
    'test_mae': stacking_test_mae
}

# BAGGING - Training
start_time = time.time()
bagging_reg.fit(x_train, y_train)
bagging_train_time = time.time() - start_time

y_pred_bagging = bagging_reg.predict(x_test)
bagging_train_mse = mean_squared_error(y_train, bagging_reg.predict(x_train))
bagging_test_mse = mean_squared_error(y_test, y_pred_bagging)
bagging_test_r2 = r2_score(y_test, y_pred_bagging)
bagging_test_mae = mean_absolute_error(y_test, y_pred_bagging)

results['Bagging'] = {
    'train_time': bagging_train_time,
    'train_mse': bagging_train_mse,
    'test_mse': bagging_test_mse,
    'test_r2': bagging_test_r2,
    'test_mae': bagging_test_mae
}

print("All models trained successfully")

All models trained successfully


In [15]:
comparison_data = [
    ['Voting', f"{voting_train_time:.4f}s", f"{voting_train_mse:.2f}", f"{voting_test_mse:.2f}", f"{voting_test_r2:.4f}", f"{voting_test_mae:.2f}"],
    ['Stacking', f"{stacking_train_time:.4f}s", f"{stacking_train_mse:.2f}", f"{stacking_test_mse:.2f}", f"{stacking_test_r2:.4f}", f"{stacking_test_mae:.2f}"],
    ['Bagging', f"{bagging_train_time:.4f}s", f"{bagging_train_mse:.2f}", f"{bagging_test_mse:.2f}", f"{bagging_test_r2:.4f}", f"{bagging_test_mae:.2f}"]
]
print("Complexity aspects of the ensemble methods:")
print(tabulate(
    comparison_data,
    headers=['Method', 'Train Time', 'Train MSE', 'Test MSE', 'Test R²', 'Test MAE'],
    tablefmt='grid'
))

Complexity aspects of the ensemble methods:
+----------+--------------+-------------+-------------+-----------+------------+
| Method   | Train Time   |   Train MSE |    Test MSE |   Test R² |   Test MAE |
+==========+==============+=============+=============+===========+============+
| Voting   | 0.0308s      | 1.21424e+11 | 1.32725e+11 |    0.588  |   294065   |
+----------+--------------+-------------+-------------+-----------+------------+
| Stacking | 2.7101s      | 9.75547e+10 | 9.48556e+10 |    0.7056 |   208090   |
+----------+--------------+-------------+-------------+-----------+------------+
| Bagging  | 0.4961s      | 0           | 2.09392e+10 |    0.935  |    75864.9 |
+----------+--------------+-------------+-------------+-----------+------------+


In [17]:
learning_comb_data = [
    ['Voting', 'Averaging/Majority Vote', 'Uniform', 'All base learners equally weighted'],
    ['Stacking', 'Meta-Learner (Sequential)', 'Learned', 'Meta-model learns optimal combination'],
    ['Bagging', 'Averaging', 'Uniform', 'All trees equally weighted']
]
print("Learning and combination strategies of the ensemble methods:")
print(tabulate(
    learning_comb_data,
    headers=['Method', 'Combination Strategy', 'Weighting', 'Description'],
    tablefmt='grid'
))

Learning and combination strategies of the ensemble methods:
+----------+---------------------------+-------------+---------------------------------------+
| Method   | Combination Strategy      | Weighting   | Description                           |
+==========+===========================+=============+=======================================+
| Voting   | Averaging/Majority Vote   | Uniform     | All base learners equally weighted    |
+----------+---------------------------+-------------+---------------------------------------+
| Stacking | Meta-Learner (Sequential) | Learned     | Meta-model learns optimal combination |
+----------+---------------------------+-------------+---------------------------------------+
| Bagging  | Averaging                 | Uniform     | All trees equally weighted            |
+----------+---------------------------+-------------+---------------------------------------+


In [18]:
# Calculate train-test gap (indicator of overfitting)
voting_gap = voting_test_mse - voting_train_mse
stacking_gap = stacking_test_mse - stacking_train_mse
bagging_gap = bagging_test_mse - bagging_train_mse

overfitting_data = [
    ['Voting', f"{voting_train_mse:.2f}", f"{voting_test_mse:.2f}", f"{voting_gap:.2f}", 'Medium'],
    ['Stacking', f"{stacking_train_mse:.2f}", f"{stacking_test_mse:.2f}", f"{stacking_gap:.2f}", 'High'],
    ['Bagging', f"{bagging_train_mse:.2f}", f"{bagging_test_mse:.2f}", f"{bagging_gap:.2f}", 'Low']
]

print("Overfitting risk assessment of the ensemble methods:")
print(tabulate(
    overfitting_data,
    headers=['Method', 'Train MSE', 'Test MSE', 'Train-Test Gap', 'Overfitting Risk'],
    tablefmt='grid'
))

Overfitting risk assessment of the ensemble methods:
+----------+-------------+-------------+------------------+--------------------+
| Method   |   Train MSE |    Test MSE |   Train-Test Gap | Overfitting Risk   |
+==========+=============+=============+==================+====================+
| Voting   | 1.21424e+11 | 1.32725e+11 |      1.13013e+10 | Medium             |
+----------+-------------+-------------+------------------+--------------------+
| Stacking | 9.75547e+10 | 9.48556e+10 |     -2.69917e+09 | High               |
+----------+-------------+-------------+------------------+--------------------+
| Bagging  | 0           | 2.09392e+10 |      2.09392e+10 | Low                |
+----------+-------------+-------------+------------------+--------------------+


In [20]:
# Performance comparison
performance_data = [
    ['Voting', f"{voting_test_mse:.2f}", f"{voting_test_mae:.2f}", f"{voting_test_r2:.4f}"],
    ['Stacking', f"{stacking_test_mse:.2f}", f"{stacking_test_mae:.2f}", f"{stacking_test_r2:.4f}"],
    ['Bagging', f"{bagging_test_mse:.2f}", f"{bagging_test_mae:.2f}", f"{bagging_test_r2:.4f}"]
]

print("Performance comparison of the ensemble methods on the test set:")
print(tabulate(
    performance_data,
    headers=['Method', 'MSE (Lower Better)', 'MAE (Lower Better)', 'R² (Higher Better)'],
    tablefmt='grid'
))

Performance comparison of the ensemble methods on the test set:
+----------+----------------------+----------------------+----------------------+
| Method   |   MSE (Lower Better) |   MAE (Lower Better) |   R² (Higher Better) |
+==========+======================+======================+======================+
| Voting   |          1.32725e+11 |             294065   |               0.588  |
+----------+----------------------+----------------------+----------------------+
| Stacking |          9.48556e+10 |             208090   |               0.7056 |
+----------+----------------------+----------------------+----------------------+
| Bagging  |          2.09392e+10 |              75864.9 |               0.935  |
+----------+----------------------+----------------------+----------------------+
